# Lección 8.6 — Detección de anomalías: Isolation Forest y compañía

> **Módulo 8 · Aprendizaje no supervisado** · Nivel 3 — Practicante
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 2.2 (atípicos), Lección 6.2 (Random Forest), Lección 8.3 (DBSCAN), Lección 4.2 (precisión y recall)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué es una anomalía y por qué detectarla es un problema **no supervisado**.
- Explicar con una analogía cómo **Isolation Forest** aísla lo raro con pocas preguntas.
- Distinguir anomalías **globales** de **locales** y saber cuándo usar **LOF**.
- Elegir `contamination` según la capacidad del negocio y evaluar el detector cuando tienes unas pocas etiquetas.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

plt.rcParams["figure.figsize"] = (8, 4.5)

def dibujar_anomalias(ax, X, es_anomalia, titulo):
    ax.scatter(*X[~es_anomalia].T, s=12, c="steelblue")
    ax.scatter(*X[es_anomalia].T, s=60, facecolors="none", edgecolors="red", linewidths=1.5)
    ax.set_title(titulo, fontsize=10)

print("✅ Todo listo")

---
## 1. 📖 La intuición: ¿a quién se aísla más rápido? 🕵️

Recuerda el juego **"¿Adivina quién?"**: 24 personajes y tienes que adivinar el de tu rival con preguntas de sí/no.

Si el personaje secreto es **el único con sombrero**, lo encuentras con **una** pregunta: "¿Tiene sombrero?". Si es uno de los muchos personajes de pelo castaño y sin lentes, necesitas **muchas** preguntas.

**Lo raro se aísla rápido.** Esa es la idea completa de **Isolation Forest**: hacer preguntas **al azar** ("¿la columna 3 es mayor que 7.2?") y contar cuántas hacen falta para dejar a cada punto **solo**. Los puntos que quedan solos con pocas preguntas son **anomalías**.

### ✋ Ejercicio 1 — Calentamiento
En una clase de 30 alumnos de 12 años, uno mide 1.95 m. Si haces preguntas al azar sobre la estatura ("¿mide más de X?", con X elegido al azar entre la estatura mínima y la máxima del grupo), ¿por qué es probable que ese alumno quede aislado en muy pocas preguntas?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Porque hay un **hueco enorme** entre él y los demás (digamos, de 1.60 a 1.95). Cualquier corte al azar que caiga en ese hueco lo separa de todos de golpe, y el hueco ocupa buena parte del rango. A un alumno de 1.48 m, rodeado de otros de 1.46, 1.47, 1.49…, solo lo aísla un corte que caiga **justo** entre sus vecinos: eso es muy improbable y necesita muchas preguntas.
</details>

---
## 2. 📖 El concepto: qué es una anomalía y por qué es no supervisado

Una **anomalía** (o atípico, *outlier*) es un dato **raro y distinto** del resto: una transacción fraudulenta, una máquina a punto de fallar, un error de captura, un paciente con una combinación de análisis extraña.

¿Por qué no usar un clasificador supervisado (Módulos 5–6)? Porque normalmente:
- Hay **poquísimos** ejemplos de anomalías (0.1 % de los datos), o **ninguno** etiquetado.
- Las anomalías de mañana pueden ser **de un tipo nuevo** que nunca has visto (un fraude inventado ayer).

Así que en vez de aprender "cómo es un fraude", aprendemos **cómo es lo normal** y marcamos lo que no encaja.

### 📖 El método más simple: el puntaje z (Lección 2.2)
Para cada columna, cuántas desviaciones típicas se aleja un valor del promedio. Si |z| > 3, es sospechoso. Funciona… **columna por columna**. Veamos su límite.

In [ ]:
rng = np.random.default_rng(0)
estatura = rng.normal(170, 8, 500)
peso = 0.9 * (estatura - 170) + 70 + rng.normal(0, 4, 500)
personas = pd.DataFrame({"estatura": estatura, "peso": peso})
personas.loc[500] = [155, 82]          # 🕵️ persona "rara" agregada a mano

z = (personas - personas.mean()) / personas.std()
print("Puntajes z de la persona 500:")
print(z.loc[500].round(2))

### 🔮 Ejercicio 2
La persona 500 mide 155 cm y pesa 82 kg. Mira sus puntajes z: ninguno pasa de 3. ¿Es una persona "normal" en estos datos? Mira la gráfica antes de responder.

✍️ *Mi respuesta:*

In [ ]:
plt.scatter(personas.estatura[:500], personas.peso[:500], s=10, alpha=.5)
plt.scatter(155, 82, s=120, facecolors="none", edgecolors="red", linewidths=2)
plt.xlabel("estatura (cm)"); plt.ylabel("peso (kg)"); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

**No es normal:** por separado, 155 cm y 82 kg son valores que se ven en estos datos, pero **la combinación** está lejísimos de la nube (alguien bajo pesa, en promedio, mucho menos). El puntaje z mira cada columna por separado y no lo detecta.

Es una **anomalía multivariada**: solo se ve mirando varias columnas a la vez. Para eso necesitamos métodos como Isolation Forest.
</details>

---
## 3. 📖 Cómo funciona Isolation Forest por dentro

1. Toma una muestra de los datos.
2. Construye un **árbol al azar**: elige una columna al azar y un punto de corte al azar entre su mínimo y su máximo; divide; repite en cada lado hasta que cada punto quede solo.
3. Anota la **profundidad** a la que quedó aislado cada punto (cuántas preguntas hicieron falta).
4. Repite con muchos árboles (un **bosque**, como en la Lección 6.2) y promedia.
5. **Pocas preguntas en promedio = muy anómalo.**

Hagamos el experimento en una dimensión, con 99 valores normales y uno extraño:

In [ ]:
rng = np.random.default_rng(1)
valores = np.append(rng.normal(0, 1, 99), 6.0)     # el último (6.0) es el extraño

def preguntas_para_aislar(datos, objetivo, rng):
    grupo, n = datos, 0
    while len(grupo) > 1:
        corte = rng.uniform(grupo.min(), grupo.max())        # pregunta al azar
        grupo = grupo[grupo <= corte] if objetivo <= corte else grupo[grupo > corte]
        n += 1
    return n

for nombre, objetivo in [("extraño (6.0)", 6.0), ("normal (cerca de 0)", valores[np.argmin(np.abs(valores))])]:
    intentos = [preguntas_para_aislar(valores, objetivo, rng) for _ in range(500)]
    print(f"{nombre:22} → preguntas promedio para aislarlo: {np.mean(intentos):.1f}")

### 🔮 Ejercicio 3
Antes de mirar la salida, ¿cuántas veces más preguntas crees que necesita el punto normal que el extraño? Después mira los resultados.

✍️ *Mi apuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

El extraño se aísla con **≈2** preguntas en promedio y el normal necesita **≈11**: unas **6 veces más**. Esa diferencia es la "puntuación de anomalía" de Isolation Forest.

Detalle bonito: Isolation Forest **no mide distancias**, solo cuenta cortes. Por eso es rápido, funciona con muchas columnas y no es tan sensible a la escala como K-means o DBSCAN (aunque escalar no hace daño).
</details>

### ▶️ Isolation Forest en 2D
Ahora con scikit-learn: dos grupos de puntos normales y 15 anomalías esparcidas. El fondo muestra la **puntuación**: cuanto más oscuro, más "normal".

In [ ]:
X_norm, _ = make_blobs(n_samples=400, centers=[[0, 0], [5, 5]], cluster_std=1.0, random_state=0)
X_raros = np.random.default_rng(2).uniform(-4, 9, size=(15, 2))
X = np.vstack([X_norm, X_raros])

bosque = IsolationForest(n_estimators=200, contamination=0.04, random_state=0).fit(X)
marcadas = bosque.predict(X) == -1          # predict: 1 = normal, -1 = anomalía

xx, yy = np.meshgrid(np.linspace(-5, 10, 200), np.linspace(-5, 10, 200))
puntuacion = bosque.score_samples(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(7, 6))
ax.contourf(xx, yy, puntuacion, levels=20, cmap="Blues")
dibujar_anomalias(ax, X, marcadas, f"Isolation Forest: {marcadas.sum()} marcadas como anomalía")
plt.show()

> 📝 `predict` devuelve **1** (normal) o **−1** (anomalía). `score_samples` devuelve una **puntuación continua**: cuanto **más baja**, más anómalo. Casi siempre te interesa más la puntuación que el sí/no: te permite **ordenar** los casos del más al menos sospechoso.

---
## 4. 📖 Hiperparámetros: ¿cuántas alarmas quieres?

| Hiperparámetro | Qué controla | Si lo **subes** ⬆️ | Si lo **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `contamination` | Qué fracción de los datos se marca como anomalía (dónde se pone el umbral) | Más alarmas: atrapas más anomalías, pero más falsas alarmas | Menos alarmas, más precisas, pero se escapan anomalías | `"auto"`, o la tasa esperada / la capacidad de revisión |
| `n_estimators` | Número de árboles | Puntuaciones más estables (más lento) | Puntuaciones más ruidosas | 100–300 |
| `max_samples` | Cuántos datos usa cada árbol | Ve más detalle, más lento | Más rápido; ayuda si hay grupos de anomalías juntas | `"auto"` (256) |
| `max_features` | Cuántas columnas puede usar cada árbol | — | Más aleatorio | 1.0 |

### 🧠 La perilla clave no cambia el modelo
`contamination` **no cambia** las puntuaciones: solo mueve el **umbral** a partir del cual se dice "anomalía". Es como el umbral de la regresión logística (Lección 4.3): el modelo ordena, tú decides dónde cortar.

### 🔮 Ejercicio 4
Usaremos `contamination` = **0.01**, **0.04** y **0.20** en los mismos datos (sabemos que hay 15 anomalías de 415 puntos, ≈3.6 %). ¿Cuántos puntos se marcarán en cada caso? ¿En cuál habrá más falsas alarmas?

✍️ *Mi apuesta:*

In [ ]:
es_raro_real = np.r_[np.zeros(400, bool), np.ones(15, bool)]      # solo para evaluar

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, cont in zip(axes, [0.01, 0.04, 0.20]):      # ✏️ prueba otros valores
    m = IsolationForest(n_estimators=200, contamination=cont, random_state=0).fit(X).predict(X) == -1
    atrapadas = (m & es_raro_real).sum()
    dibujar_anomalias(ax, X, m, f"contamination={cont}: {m.sum()} alarmas\n"
                                f"{atrapadas} de 15 anomalías atrapadas · {m.sum() - atrapadas} falsas alarmas")
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **0.01** → 5 alarmas, **todas** anomalías reales (0 falsas alarmas), pero se escapan 10 de las 15.
- **0.04** → 17 alarmas: atrapa **8** de las 15, con 9 falsas alarmas (puntos de la orilla de los grupos).
- **0.20** → 83 alarmas: atrapa **13** de las 15, pero con 70 falsas alarmas: marca toda la orilla de los grupos normales.

Es el compromiso **precisión vs. recall** de la Lección 4.2. ¿Y las anomalías que nunca se atrapan? Las 15 se generaron **al azar** en todo el cuadrado, así que varias cayeron **dentro o muy cerca** de los grupos normales: ¿son de verdad anomalías? Ningún método las encontraría, y no debería.

En la vida real casi nunca sabes la tasa real. Una regla práctica: **pon `contamination` según cuántos casos puede revisar tu equipo** (si revisan 50 de 10,000 al día, 0.005).
</details>

---
## 5. 📖 ¿Qué pasa si… la anomalía es local? Entra LOF

Isolation Forest piensa en términos **globales**: ¿qué tan fácil es aislar este punto en **todo** el mapa? Pero a veces una anomalía solo se nota **comparada con sus vecinos**.

Ejemplo: una ciudad con un barrio **muy denso** (edificios) y uno **disperso** (casas de campo). Una casa a 200 m del edificio más cercano es rarísima en el barrio denso, pero normal en el campo.

**LOF** (*Local Outlier Factor*, factor de atipicidad local) compara la **densidad** alrededor de cada punto con la densidad alrededor de **sus vecinos**. Si tus vecinos viven apretados y tú estás solo, eres raro.

### 🔮 Ejercicio 5
Tenemos un grupo **muy denso**, un grupo **disperso** y un punto rojo **justo al lado** del grupo denso (a una distancia que sería normal en el grupo disperso). ¿Quién lo detecta: Isolation Forest, LOF, ambos o ninguno?

✍️ *Mi apuesta:*

In [ ]:
rng = np.random.default_rng(3)
denso = rng.normal([0, 0], 0.3, (200, 2))
disperso = rng.normal([6, 6], 2.0, (200, 2))
sospechoso = np.array([[1.2, 1.2]])                     # cerca del grupo denso, pero fuera de él
X_loc = np.vstack([denso, disperso, sospechoso])

iso = IsolationForest(contamination=0.02, random_state=0).fit(X_loc)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.02)
marcas_lof = lof.fit_predict(X_loc) == -1
marcas_iso = iso.predict(X_loc) == -1
# Puesto del punto sospechoso en el ranking de sospecha (1 = el más sospechoso de los 401)
puesto_iso = (iso.score_samples(X_loc) < iso.score_samples(X_loc)[-1]).sum() + 1
puesto_lof = (lof.negative_outlier_factor_ < lof.negative_outlier_factor_[-1]).sum() + 1

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, marcas, nombre, puesto in [(axes[0], marcas_iso, "Isolation Forest", puesto_iso),
                                   (axes[1], marcas_lof, "LOF", puesto_lof)]:
    ax.scatter(*sospechoso.T, marker="*", s=150, c="gold", edgecolors="k", zorder=3)
    dibujar_anomalias(ax, X_loc, marcas, f"{nombre}: ¿marca la estrella? {'SÍ' if marcas[-1] else 'NO'} "
                                         f"(puesto {puesto} de {len(X_loc)})")
plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **Isolation Forest no la marca (NO, puesto ≈11):** gasta sus 8 alarmas en los puntos de la orilla del grupo disperso, que están más "solos" en términos absolutos. Visto desde el mapa completo, la estrella está en una zona bastante poblada.
- **LOF sí la marca (SÍ, puesto 1: ¡la más sospechosa de todas!):** sus vecinos (el grupo denso) viven apretadísimos y la estrella está sola en comparación. **En su barrio**, es rarísima.

Regla práctica: **Isolation Forest** para anomalías globales y muchos datos/columnas; **LOF** cuando las densidades varían y lo que importa es ser raro "en tu barrio". Si dudas, prueba los dos y compara.
</details>

### 📖 La familia completa ("y compañía")

| Método | Idea | Cuándo usarlo |
|---|---|---|
| **Puntaje z / rango intercuartílico** (Lección 2.2) | Valor extremo en **una** columna | Primer filtro rápido; errores de captura |
| **Isolation Forest** | Lo raro se aísla con pocos cortes al azar | Punto de partida general: rápido, muchas columnas |
| **LOF** (`LocalOutlierFactor`) | Densidad local menor que la de tus vecinos | Grupos de densidades distintas; anomalías "locales" |
| **DBSCAN / HDBSCAN** (Lección 8.3) | Lo que no pertenece a ninguna zona densa = ruido | Ya estás agrupando y quieres el ruido "gratis" |
| **One-Class SVM** (`OneClassSVM`) | Dibuja una frontera alrededor de lo normal (Lección 5.5) | Pocos datos, datos de entrenamiento **limpios** |
| **Envolvente elíptica** (`EllipticEnvelope`) | Ajusta una "elipse" a los datos (supone forma de campana) | Datos aproximadamente normales (gaussianos) |
| **Error de reconstrucción** (PCA, autoencoders, Lección 13.2) | Lo normal se comprime y reconstruye bien; lo raro, no | Imágenes, señales, muchas columnas correlacionadas |

### ✋ Ejercicio 6 — Elige el método
1. Una fábrica tiene 6 meses de datos de sensores **sin ninguna falla** y quiere una alarma para cuando algo se salga de lo normal.
2. Un banco tiene 10 millones de transacciones con 30 columnas y quiere una lista diaria de las 500 más sospechosas.
3. Una tabla de clientes tiene una edad de 999 años.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Es **detección de novedades** (*novelty detection*): entrenas solo con datos **limpios** y vigilas los nuevos. One-Class SVM, Isolation Forest o LOF con `novelty=True` (para poder usar `predict` con datos nuevos).
2. **Isolation Forest**: escala bien a millones de filas y muchas columnas. Ordenar por `score_samples` y tomar las 500 más bajas.
3. Un simple **rango válido** o puntaje z: es un **error de captura** (Lección 2.2), no hace falta ML. Corrígelo antes de cualquier otro análisis.
</details>

---
## 6. 📖 ¿Y cómo sé si mi detector funciona?

Sin etiquetas, no hay exactitud que calcular. Pero casi siempre puedes conseguir **algunas**:
- **Revisión humana de los primeros casos:** un analista revisa los 100 más sospechosos. ¿Cuántos eran de verdad problemas? Eso es la **precisión en los primeros k** (*precision@k*).
- **Casos conocidos del pasado:** si tienes 30 fraudes confirmados del año pasado, ¿en qué posición del ranking quedan? Eso estima el **recall**.

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Las alarmas son siempre "los clientes más grandes" | Una columna enorme domina, o se marcan **valores extremos legítimos** | Escalar, transformar (logaritmo), quitar columnas de tamaño o normalizarlas |
| Demasiadas alarmas que el equipo no alcanza a revisar | `contamination` muy alto | Fijarlo según la capacidad de revisión; ordenar por puntuación |
| Anomalías obvias que no se detectan | Son locales (Isolation Forest) o forman un **grupo** de anomalías juntas | LOF; bajar `max_samples` |
| LOF marca puntos rarísimos en un grupo y nada en otro | Datos sin escalar (LOF usa distancias) | `StandardScaler` |
| "Anomalía = error", y se borran | Una anomalía puede ser **el dato más valioso** (fraude, falla, cliente VIP) | Investigar antes de borrar |
| El detector empeora con el tiempo | Lo "normal" cambió (Lección 15.2, deriva de datos) | Reentrenar periódicamente con datos recientes |

### 🔮 Ejercicio 7
Un detector de fraude marca 200 transacciones al día con `contamination=0.002`. El equipo revisa todas y encuentra 6 fraudes reales. ¿Cuál es la precisión del detector? ¿Es un mal detector?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Precisión = 6 / 200 = **3 %**. Parece terrible, pero depende de la **tasa base**: si el fraude ocurre en 1 de cada 10,000 transacciones (0.01 %), revisar al azar daría 0.01 % de precisión, y el detector es **300 veces mejor** que el azar. Con anomalías, compara siempre contra la tasa base (Lección 4.7, líneas base) y contra el costo: ¿cuánto cuesta revisar 200 casos frente a lo que ahorran 6 fraudes detectados?
</details>

---
## 7. 📋 Fichas de los modelos

| | Isolation Forest | LOF (Local Outlier Factor) |
|---|---|---|
| **Tipo** | No supervisado: detección de anomalías | Igual |
| **Intuición** | "¿Adivina quién?": lo raro se aísla con pocas preguntas al azar | "Raro en tu barrio": estás más solo que tus vecinos |
| **Cómo funciona** | Bosque de árboles con cortes al azar; puntuación = profundidad promedio para aislar | Compara la densidad local de cada punto con la de sus k vecinos |
| **Hiperparámetros clave** | `contamination`, `n_estimators`, `max_samples` | `n_neighbors`, `contamination`, `novelty` |
| **✅ Fortalezas** | Rápido, escala a millones · Muchas columnas · Poco sensible a la escala · Da puntuación para ordenar | Detecta anomalías **locales** · Maneja densidades distintas |
| **❌ Debilidades** | Le cuestan las anomalías locales y los grupos de anomalías · Cortes solo paralelos a los ejes | Lento con muchos datos · Necesita escalar · Para datos nuevos hay que usar `novelty=True` |
| **Úsalo cuando…** | Necesitas un detector general y rápido | Los grupos tienen densidades distintas o importa el contexto local |

## 8. 🧠 Resumen en 5 puntos
1. Detectar anomalías es aprender **cómo es lo normal** y marcar lo que no encaja; casi nunca hay etiquetas suficientes para hacerlo supervisado.
2. El puntaje z mira columnas por separado; las anomalías **multivariadas** necesitan métodos como Isolation Forest o LOF.
3. **Isolation Forest**: lo raro se aísla con **pocos cortes al azar**. **LOF**: lo raro es **menos denso que sus vecinos**.
4. `contamination` solo mueve el **umbral**: elígelo según la capacidad de revisión; trabaja con la **puntuación** para ordenar.
5. Evalúa con unas pocas etiquetas (precisión en los primeros k, recall de casos conocidos) y compara contra la **tasa base**.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas** (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** En Isolation Forest, un punto que se aísla con **muy pocos** cortes es…
- a) Muy normal · b) Muy anómalo · c) Un centroide · d) Un error del algoritmo

**2.** `IsolationForest(...).predict(X)` devuelve −1 para un punto. ¿Qué significa?
- a) Normal · b) Anomalía · c) Sin decidir · d) Ruido de DBSCAN

**3.** Subes `contamination` de 0.01 a 0.10. ¿Qué pasa con las puntuaciones de `score_samples`?
- a) Bajan todas · b) Suben todas · c) No cambian; solo cambia el umbral · d) Se invierten

**4.** ¿Cuándo es preferible LOF a Isolation Forest?
- a) Con 50 millones de filas · b) Cuando hay grupos con densidades muy distintas · c) Cuando no hay columnas numéricas · d) Nunca

**5.** Una persona mide 1.55 m y pesa 82 kg; ninguno de los dos valores tiene |z| > 3. ¿Qué tipo de anomalía es?
- a) No es anomalía · b) Multivariada · c) Un error de captura seguro · d) Local de DBSCAN

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Pocas preguntas para aislarlo = raro.
2. **b)** −1 = anomalía, 1 = normal.
3. **c)** El modelo es el mismo; solo se mueve el umbral de decisión.
4. **b)** LOF compara cada punto con su vecindario.
5. **b)** La rareza está en la **combinación** de columnas.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Un hospital usa Isolation Forest sobre los análisis de sangre de sus pacientes y la mayoría de las alarmas son pacientes de la unidad de cuidados intensivos. ¿Está fallando el detector? ¿Qué cambiarías?

**B. Caso de análisis 💼** Una tienda en línea quiere detectar reseñas falsas (bots). Tiene 2 millones de reseñas sin etiquetar y 300 reseñas que el equipo ya confirmó como falsas. Propón un plan: qué método usarías, cómo elegirías el umbral y cómo usarías las 300 reseñas confirmadas.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** No está fallando: esos pacientes **son** estadísticamente raros comparados con el hospital completo. Pero no es lo que se busca (probablemente se quieren detectar errores de laboratorio o casos inesperados). Opciones: entrenar un detector **por unidad** o por tipo de paciente (lo "normal" depende del contexto: anomalía **contextual**), agregar columnas de contexto, o usar LOF para comparar a cada paciente con sus pares.

**B.** Plan: (1) crear columnas útiles (reseñas por usuario y día, longitud, antigüedad de la cuenta, similitud entre reseñas…); (2) Isolation Forest para **ordenar** las 2 millones por puntuación; (3) umbral según la capacidad del equipo de moderación (p. ej. revisan 1,000 al día → las 1,000 más sospechosas); (4) usar las **300 confirmadas** para **evaluar**: ¿qué porcentaje cae en el 1 % más sospechoso? (recall); comparar configuraciones con esa métrica. Con el tiempo, cuando haya miles de etiquetas, podrías pasar a un modelo **supervisado** (o combinar ambos).
</details>

### 9.3 🧩 Práctica integradora: "El detector de fraude"

Un banco tiene 5,000 transacciones con tarjeta de un día. Columnas: **monto** (pesos), **hora** (0–23), **distancia de casa** (km) y **compras en la última hora**. El banco sospecha que hay fraude, pero **no tiene etiquetas**. El equipo de seguridad puede revisar **50 transacciones** al día.

> 🔒 Esta vez sí tenemos la verdad escondida (`es_fraude`), **solo** para evaluar al final. Los detectores no la ven.

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
rng = np.random.default_rng(42)
n_ok, n_fraude = 4_950, 50
normales = pd.DataFrame({
    "monto":            rng.lognormal(6.0, 0.9, n_ok).round(0),        # casi todas de cientos de pesos, algunas de miles
    "hora":             rng.normal(14, 4, n_ok).clip(0, 23).round(0),
    "distancia_km":     rng.exponential(5, n_ok).round(1),
    "compras_ult_hora": rng.poisson(0.3, n_ok),
})
fraudes = pd.DataFrame({
    "monto":            rng.lognormal(7.3, 0.6, n_fraude).round(0),     # montos altos, pero no siempre exagerados
    "hora":             rng.choice([1, 2, 3, 4, 23], n_fraude).astype(float),   # de madrugada
    "distancia_km":     rng.uniform(40, 400, n_fraude).round(1),        # lejos de casa
    "compras_ult_hora": rng.poisson(3, n_fraude),                       # varias compras seguidas
})
transacciones = pd.concat([normales, fraudes], ignore_index=True)
es_fraude = np.r_[np.zeros(n_ok, bool), np.ones(n_fraude, bool)]        # 🔒 solo para evaluar
orden = rng.permutation(len(transacciones))
transacciones, es_fraude = transacciones.iloc[orden].reset_index(drop=True), es_fraude[orden]

print(f"Transacciones: {len(transacciones)} · fraudes escondidos: {es_fraude.sum()} ({es_fraude.mean():.0%})")
transacciones.describe().round(1)

**Paso 2 🔮 — La línea base ingenua.** La primera idea de cualquiera: revisar las **50 transacciones de mayor monto**. ¿Cuántos de los 50 fraudes crees que atrapará?

✍️ *Mi apuesta:*

In [ ]:
REVISIONES = 50
top_monto = transacciones["monto"].nlargest(REVISIONES).index
print(f"Fraudes atrapados revisando los {REVISIONES} montos más altos: {es_fraude[top_monto].sum()} de {es_fraude.sum()}")

<details><summary><b>👀 Ver respuesta</b></summary>

Solo **3 de 50**. Las transacciones legítimas grandes (una compra de muebles, un boleto de avión) son mucho más numerosas que los fraudes, y muchos fraudes tienen montos "normales" (unos miles de pesos). Mirar una sola columna no basta: el fraude está en la **combinación** (monto alto **y** de madrugada **y** lejos **y** varias compras seguidas).
</details>

**Paso 3 ✋ — Prepara los datos.** `monto` y `distancia_km` tienen colas muy largas (unos pocos valores enormes). ¿Qué transformación de la Lección 2.3 usarías antes de LOF? ¿Hace falta escalar?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un **logaritmo** (`np.log1p`) en `monto` y `distancia_km`, para que los valores enormes no dominen, y después **escalar** todo con `StandardScaler`, porque LOF usa distancias. Isolation Forest es menos sensible, pero usar los mismos datos preparados para los dos nos permite compararlos de forma justa.
</details>

In [ ]:
X_tx = transacciones.copy()
X_tx[["monto", "distancia_km"]] = np.log1p(X_tx[["monto", "distancia_km"]])   # ✏️ (ya está; compruébalo)
X_tx = StandardScaler().fit_transform(X_tx)
print("✅ Datos preparados:", X_tx.shape)

**Paso 4 🔮 — Isolation Forest y LOF.** Ordenamos las transacciones por puntuación de anomalía y revisamos las **50 más sospechosas** de cada método. ¿Cuál atrapará más fraudes? ¿Le ganarán a los 3 de la línea base?

✍️ *Mi apuesta:*

In [ ]:
iso_tx = IsolationForest(n_estimators=300, random_state=0).fit(X_tx)
puntaje_iso = -iso_tx.score_samples(X_tx)                 # le damos la vuelta: más alto = más sospechoso

N_VECINOS = 30      # ✏️ después prueba 60
lof_tx = LocalOutlierFactor(n_neighbors=N_VECINOS).fit(X_tx)
puntaje_lof = -lof_tx.negative_outlier_factor_            # igual: más alto = más sospechoso

for nombre, puntaje in [("Isolation Forest", puntaje_iso), ("LOF", puntaje_lof)]:
    top = np.argsort(puntaje)[::-1][:REVISIONES]
    print(f"{nombre:17} → fraudes en las {REVISIONES} más sospechosas: {es_fraude[top].sum()}  "
          f"(precisión = {es_fraude[top].mean():.0%})")

<details><summary><b>👀 Ver respuesta</b></summary>

- **Isolation Forest:** atrapa **47 de 50** fraudes (precisión 94 %). ¡Más de **15 veces mejor** que la línea base, sin una sola etiqueta!
- **LOF** (con 30 vecinos): solo **15**. ¿Por qué tan mal? Los 50 fraudes se **parecen entre sí** (madrugada, lejos, varias compras): forman su **propio grupito**. Para LOF, cada fraude tiene a otros fraudes como vecinos, con una densidad parecida a la suya… así que "en su barrio" no es raro.

✏️ Cambia `N_VECINOS = 60` (más vecinos que fraudes) y vuelve a ejecutar: LOF sube a **34**, porque ahora el vecindario de cada fraude incluye por fuerza transacciones normales. Es la fila de "grupos de anomalías juntas" de la tabla de errores comunes. Aquí Isolation Forest gana porque los fraudes son anomalías **globales** (están lejos de casi todo), que es justo su especialidad.
</details>

**Paso 5 ✏️ — ¿Y si el equipo creciera?** Cambia `REVISIONES` a 25, 100 y 200 y vuelve a ejecutar la celda de abajo. ¿Cómo cambian los fraudes atrapados y la precisión?

In [ ]:
REVISIONES = 100     # ✏️ prueba 25, 50, 100, 200
top = np.argsort(puntaje_iso)[::-1][:REVISIONES]
print(f"Revisando {REVISIONES}: atrapamos {es_fraude[top].sum()} de {es_fraude.sum()} fraudes "
      f"(recall = {es_fraude[top].sum() / es_fraude.sum():.0%}, precisión = {es_fraude[top].mean():.0%})")

<details><summary><b>👀 Ver respuesta</b></summary>

Con estos datos:

| Revisiones | Fraudes atrapados | Recall | Precisión |
|---|---|---|---|
| 25 | 25 | 50 % | 100 % |
| 50 | 47 | 94 % | 94 % |
| 100 | 50 | 100 % | 50 % |
| 200 | 50 | 100 % | 25 % |

Duplicar el equipo (de 50 a 100 revisiones) atrapa solo 3 fraudes más, y de 100 a 200 no atrapa ninguno. Es la decisión de negocio clásica: **¿cuánto vale cada fraude atrapado frente al costo de revisar?** (Lección 4.8.)
</details>

**Paso 6 ✋ — El informe.** Escribe tres frases para el director de seguridad: qué propones, qué resultado esperar y qué riesgo tiene.

✍️ *Mi informe:*

<details><summary><b>👀 Ver respuesta</b></summary>

*"Propongo revisar cada día las 50 transacciones que un detector de anomalías (Isolation Forest) marca como más sospechosas, en lugar de las 50 de mayor monto. En una simulación atrapó a 47 de cada 50 fraudes, contra 3 del método actual, sin necesitar fraudes etiquetados. Riesgo: detecta lo **inusual**, así que un fraude que imite el comportamiento normal se escaparía, y lo 'normal' cambia con el tiempo (temporadas, nuevos clientes): hay que reentrenarlo con frecuencia y usar los fraudes confirmados por el equipo para medir su desempeño cada mes."*
</details>

---
## 📚 Para profundizar
- 📄 **Artículo original:** Liu, Ting y Zhou (2008), *Isolation Forest*: corto y muy legible.
- 📺 **Video:** "Isolation Forest — Anomaly Detection" y StatQuest sobre LOF (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Novelty and Outlier Detection*: incluye una comparación visual de todos los métodos.

---
⬅️ **Anterior:** [8.5 Visualización: t-SNE y UMAP, el plano de mesas de una boda](8.5-tsne-y-umap.ipynb) · ➡️ **Siguiente:** [8.7 Sistemas de recomendación: "gente como tú también disfrutó…"](8.7-sistemas-de-recomendacion.ipynb)